# Memory AI Lab — TCN Boundary Detector : Pre-training SuperDialseg

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Stratégie de pré-entraînement

```
Phase 1 — Pré-entraînement SuperDialseg   (127K msgs, ~25% boundaries)
   → Le TCN apprend à détecter les changements de topic en général
   → Dataset 22× plus grand que notre tune_early

Phase 2 — Fine-tuning tune_early          (5820 msgs, ~5% boundaries)
   → Adaptation à nos données (FR/EN, style WhatsApp, gaps réels)
   → LR réduit (1e-4) — préserve les features apprises en phase 1

Hypothèse : le pré-entraînement améliore la précision Stage 1
   actuelle ≈ 0.60 → cible ≥ 0.70, ce qui devrait gagner +0.03 ARI
```

## Données requises sur Google Drive (`memory_ai_data/`)

```
memory_ai_data/
├── group_anon.txt
├── group_gold_tune.json
├── group_gold_test.json
└── superdialgseg/
    └── superseg/
        ├── segmentation_file_train.json    ← phase 1
        └── segmentation_file_test.json     ← évaluation optionnelle
```

Le modèle est sauvegardé dans `memory_ai_data/boundary_detector_tcn.pt`.

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale ────────────────────────────────
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR  = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

# Données WhatsApp + gold
FILES_TO_COPY = [
    'group_anon.txt',
    'group_gold_tune.json',
    'group_gold_test.json',
    'group_embeddings_me5.npy',
]
for fname in FILES_TO_COPY:
    src = f'{DRIVE_DIR}/{fname}'
    dst = f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie {fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  {fname} absent sur Drive (ignoré)')

# SuperDialseg — copié seulement si absent (fichier ~300MB)
SD_DRIVE = f'{DRIVE_DIR}/superdialgseg/superseg'
SD_LOCAL = '/content/superdialgseg'
os.makedirs(SD_LOCAL, exist_ok=True)

for fname in ['segmentation_file_train.json', 'segmentation_file_test.json']:
    src = f'{SD_DRIVE}/{fname}'
    dst = f'{SD_LOCAL}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie SuperDialseg/{fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  SuperDialseg/{fname} déjà en local ✓')
    else:
        print(f'  ⚠️  SuperDialseg/{fname} absent — vérifier le chemin Drive')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Embeddings mE5-base — données WhatsApp (cache) ────────────
import numpy as np
import torch
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

MODEL_NAME  = 'intfloat/multilingual-e5-base'
PREFIX      = 'passage: '
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [PREFIX + a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings_me5.npy'
    print('[2/2] Embeddings WhatsApp chargés depuis cache')
else:
    print(f'[2/2] Calcul embeddings WhatsApp sur {device} ...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    all_embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')

print(f'      Shape : {all_embeddings.shape}')

In [ ]:
# ── CELLULE 5 : Charger tune_early et test (protocole B) ──────────────────
TUNE_SPLIT = 0.65   # identique dans 01_eval_ari.ipynb et 02_train_boundary_detector.ipynb

def load_gold(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts_all, y_true_tune_all, tune_eps_all, tune_meta = load_gold(f'{DATA_DIR}/group_gold_tune.json')
test_arts,     y_true_test,     test_eps,     test_meta  = load_gold(f'{DATA_DIR}/group_gold_test.json')

n_tune  = len(tune_arts_all)
n_split = int(n_tune * TUNE_SPLIT)

arts_early = all_artifacts[:n_split]
emb_early  = all_embeddings[:n_split]
y_early    = y_true_tune_all[:n_split]

arts_test  = all_artifacts[n_tune:n_tune + len(test_arts)]
emb_test   = all_embeddings[n_tune:n_tune + len(test_arts)]

print(f'Tune early : {n_split} msgs → fine-tuning')
print(f'Test       : {len(test_arts)} msgs → évaluation finale')

In [ ]:
# ── CELLULE 6 : Embeddings SuperDialseg (mE5-base, ~127K msgs) ────────────
# Durée estimée : ~8 min sur GPU T4
# Cache sauvegardé localement — regénéré si absent
from superdialgseg_loader import load_superseg, print_stats

SD_TRAIN_JSON = '/content/superdialgseg/segmentation_file_train.json'
SD_EMB_CACHE  = '/content/superdialgseg_embeddings.npy'
SD_CACHE_DRIVE = f'{DRIVE_DIR}/superdialgseg_embeddings.npy'

# Charger le dataset SuperDialseg
print('Chargement SuperDialseg train...')
sd_utterances, sd_y_true, sd_artifacts = load_superseg(SD_TRAIN_JSON)
print_stats(sd_y_true, label='SuperDialseg train')

# Embeddings — essayer de récupérer depuis Drive d'abord
import shutil
if not os.path.exists(SD_EMB_CACHE) and os.path.exists(SD_CACHE_DRIVE):
    print('  Copie cache embeddings SuperDialseg depuis Drive...')
    shutil.copy2(SD_CACHE_DRIVE, SD_EMB_CACHE)

if os.path.exists(SD_EMB_CACHE):
    sd_embeddings = np.load(SD_EMB_CACHE)
    assert len(sd_embeddings) == len(sd_utterances), 'Cache SuperDialseg périmé — supprimer'
    print(f'Embeddings SuperDialseg chargés depuis cache — shape : {sd_embeddings.shape}')
else:
    print(f'Calcul embeddings SuperDialseg ({len(sd_utterances)} msgs) sur {device}...')
    print('  (durée ~8 min GPU T4 — à sauvegarder sur Drive après)')

    # Charger le modèle si pas encore chargé
    if 'model' not in dir():
        model = SentenceTransformer(MODEL_NAME, device=device)

    sd_texts = [PREFIX + u for u in sd_utterances]
    sd_embeddings = model.encode(
        sd_texts, batch_size=512, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)

    np.save(SD_EMB_CACHE, sd_embeddings)
    print(f'✓ Sauvegardé localement → {SD_EMB_CACHE}')

    # Copier sur Drive pour éviter de recalculer la prochaine fois
    shutil.copy2(SD_EMB_CACHE, SD_CACHE_DRIVE)
    print(f'✓ Copié sur Drive → {SD_CACHE_DRIVE}')

print(f'\nShape embeddings SuperDialseg : {sd_embeddings.shape}')

In [ ]:
# ── CELLULE 7 : Phase 1 — Pré-entraînement sur SuperDialseg ───────────────
# ~127K messages, ~25% frontières — grande diversité thématique
# Objectif : le TCN apprend le signal de changement de topic en général
#
# Note sur la différence de distribution :
#   SuperDialseg : 25% frontières, timestamps synthétiques (1min/msg)
#   Nos données  :  5% frontières, timestamps réels (gaps variables)
#   → Le fine-tuning phase 2 corrige cette distribution shift
#
# Hyperparamètres phase 1 :
#   - lr=1e-3 (standard)
#   - 20 epochs suffisent (dataset 22x plus grand)
#   - focal_gamma=2.0 (imbalance 25% → 3:1, moins sévère que nos 5%)
#   - stride_min=25 (large stride → bonne couverture sans overfitting)
from boundary_detector_tcn import TCNBoundaryDetector

detector = TCNBoundaryDetector(
    device      = device,
    channels    = 128,
    n_blocks    = 3,
    kernel_size = 3,
    dropout     = 0.15,
    window_size = 100,
)

print('═' * 60)
print('Phase 1 — Pré-entraînement SuperDialseg')
print(f'  {len(sd_utterances):,} messages · ~{int(len(sd_utterances)*0.25):,} frontières')
print('═' * 60)

detector.fit_sequence(
    embeddings  = sd_embeddings,
    artifacts   = sd_artifacts,
    y_true      = sd_y_true,
    n_epochs    = 20,
    lr          = 1e-3,
    weight_decay= 1e-4,
    batch_size  = 32,
    focal_gamma = 2.0,
    stride_min  = 25,
    stride_max  = 50,
    val_split   = 0.05,   # 5% validation — ~6K msgs
    verbose     = True,
)

print('\n✓ Phase 1 terminée')

In [ ]:
# ── CELLULE 8 : Phase 2 — Fine-tuning sur tune_early ──────────────────────
# LR réduit (1e-4) → adaptation sans détruire les features pré-entraînées
# Moins d'epochs (40) car dataset petit → convergence rapide
# stride_min=10 → dense augmentation comme dans 02_train_boundary_detector.ipynb
from boundary_detector_tcn import extract_boundary_labels

y_early_bin = extract_boundary_labels(y_early, len(arts_early))
n_pos = int(y_early_bin.sum())
n_neg = len(y_early_bin) - n_pos
print(f'tune_early : {n_pos} frontières · {n_neg} continuations (ratio 1:{n_neg//max(n_pos,1)})')

print()
print('═' * 60)
print('Phase 2 — Fine-tuning tune_early')
print(f'  {len(arts_early):,} messages · {n_pos} frontières (~{100*n_pos/len(arts_early):.1f}%)')
print('═' * 60)

detector.fit_sequence(
    embeddings  = emb_early,
    artifacts   = arts_early,
    y_true      = y_early,
    n_epochs    = 40,
    lr          = 1e-4,     # LR réduit — fine-tuning
    weight_decay= 1e-4,
    batch_size  = 32,
    focal_gamma = 2.0,
    stride_min  = 10,       # dense augmentation
    stride_max  = 25,
    val_split   = 0.10,
    verbose     = True,
)

print('\n✓ Phase 2 terminée')

In [ ]:
# ── CELLULE 9 : Optimisation seuil sur tune_early (recall ≥ 0.85) ──────────
MIN_RECALL = 0.85

detector.optimize_threshold_sequence(
    embeddings = emb_early,
    artifacts  = arts_early,
    y_true     = y_early,
    min_recall = MIN_RECALL,
)

print(f'\nSeuil retenu : {detector.threshold:.3f}')

In [ ]:
# ── CELLULE 10 : Évaluation sur TEST ──────────────────────────────────────
# Comparaison avec la baseline TCN (sans pré-entraînement) : prec≈0.60
from boundary_detector_tcn import extract_boundary_labels

y_test_bin = extract_boundary_labels(y_true_test, len(arts_test))

probs_test = detector.predict_proba_sequence(emb_test, arts_test)
preds_test = (probs_test >= detector.threshold).astype(int)

tp   = int(((preds_test == 1) & (y_test_bin == 1)).sum())
fp   = int(((preds_test == 1) & (y_test_bin == 0)).sum())
fn   = int(((preds_test == 0) & (y_test_bin == 1)).sum())
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)
n_b  = int(preds_test.sum())
n_g  = int(y_test_bin.sum())

print(f"""
╔══ TCN + SuperDialseg PRE-TRAINING — Résultat TEST ════╗
║  Précision  : {prec:.4f}   (baseline sans pré-entr ≈ 0.60)  ║
║  Rappel     : {rec:.4f}                                ║
║  F1         : {f1:.4f}                                 ║
╠═══════════════════════════════════════════════════════╣
║  Frontières : {n_b} prédit / {n_g} gold                ║
║  Seuil      : {detector.threshold:.3f}                 ║
╚═══════════════════════════════════════════════════════╝

→ Précision >> 0.60 : pré-entraînement utile → gain ARI attendu
→ Précision ≈  0.60 : pré-entraînement neutre (distribution trop différente)
→ Précision <  0.60 : réduire n_epochs phase 1 ou lr phase 2
""")

In [ ]:
# ── CELLULE 11 : Sauvegarde (local + Drive) ────────────────────────────────
import shutil

LOCAL_PATH = f'{DATA_DIR}/boundary_detector_tcn.pt'
DRIVE_PATH = f'{DRIVE_DIR}/boundary_detector_tcn.pt'

detector.save(LOCAL_PATH)
shutil.copy2(LOCAL_PATH, DRIVE_PATH)

print(f'✓ TCN pré-entraîné copié vers Drive → {DRIVE_PATH}')
print(f'  Seuil : {detector.threshold:.3f}')
print()
print('Étape suivante : ouvrir 01_eval_ari.ipynb')
print('  USE_HYBRID=True → chargera boundary_detector_tcn.pt en priorité')
print('  Comparer ARI avec le TCN entraîné sans pré-entraînement (02_train_...)')